# Mastermind

Mastermind is a classic code-breaking game:

- The **codemaker** picks a secret code of 4 pegs, each peg one of 6 colors (repeats allowed) — 1296 possible codes.
- The **codebreaker** makes guesses. After each guess they're told:
  - **black pegs**: number of pegs that are the right color in the right position.
  - **white pegs**: number of pegs that are the right color but wrong position (excluding anything already counted as black).
- The codebreaker wins by guessing the exact code, ideally in as few guesses as possible.

This notebook has two parts:

1. A **playable version** — you are the codebreaker, the computer picks the secret.
2. An automated **solver** using Knuth's minimax strategy, which can solve any secret in at most 5 guesses.

In [ ]:
import random
import itertools
from collections import Counter

COLORS = "RGBYOP"  # Red, Green, Blue, Yellow, Orange, Purple
CODE_LENGTH = 4
ALL_CODES = list(itertools.product(COLORS, repeat=CODE_LENGTH))


def score(guess, secret):
    """Return (black, white) pegs for guess against secret."""
    black = sum(g == s for g, s in zip(guess, secret))

    guess_counts = Counter(guess)
    secret_counts = Counter(secret)
    total_matches = sum((guess_counts & secret_counts).values())

    white = total_matches - black
    return black, white


def random_secret():
    return tuple(random.choice(COLORS) for _ in range(CODE_LENGTH))


def format_code(code):
    return "".join(code)

## Play the game

Run the cell below. Enter guesses as 4 letters from `R G B Y O P` (e.g. `RGBY`). You have 10 tries.

In [ ]:
def play(max_tries=10):
    secret = random_secret()
    print(f"I've picked a secret code of {CODE_LENGTH} colors from {list(COLORS)}.")
    print(f"You have {max_tries} tries. Good luck!\n")

    attempt = 1
    while attempt <= max_tries:
        raw = input(f"Guess {attempt}/{max_tries}: ").strip().upper()

        if len(raw) != CODE_LENGTH or any(c not in COLORS for c in raw):
            print(f"  Invalid guess — enter exactly {CODE_LENGTH} letters from {list(COLORS)} (e.g. RGBY). Doesn't count as a try.")
            continue

        guess = tuple(raw)
        black, white = score(guess, secret)
        print(f"  -> {black} black, {white} white")

        if black == CODE_LENGTH:
            print(f"\nYou win in {attempt} guesses! The code was {format_code(secret)}.")
            return

        attempt += 1

    print(f"\nOut of tries — the secret code was {format_code(secret)}.")


play()

## Auto-solver (Knuth's minimax algorithm)

Donald Knuth showed in 1977 that Mastermind can always be solved in at most 5 guesses using this strategy:

1. Start with a fixed first guess (`AABB`-style, e.g. `RRGG`).
2. Keep a set `S` of all codes still consistent with every (guess, score) pair seen so far.
3. For the next guess, consider every possible code (not just those in `S`). For each candidate, compute how it would partition `S` by the score it would produce against each member of `S`, and take the size of the **largest** resulting partition (the worst case).
4. Pick the candidate that **minimizes** that worst case (minimax), preferring a candidate already in `S` on ties (so a lucky guess can still win outright).
5. Repeat until `S` has one element — the secret.

In [ ]:
def solve(secret, verbose=False):
    """Solve for `secret` using Knuth's minimax strategy. Returns the list of guesses made."""
    possible = list(ALL_CODES)
    guess = ("R", "R", "G", "G")
    guesses_made = []

    while True:
        black, white = score(guess, secret)
        guesses_made.append(guess)
        if verbose:
            print(f"  guess {format_code(guess)} -> {black} black, {white} white")

        if black == CODE_LENGTH:
            return guesses_made

        # Keep only codes consistent with this guess/score.
        possible = [c for c in possible if score(guess, c) == (black, white)]

        if len(possible) == 1:
            guess = possible[0]
            continue

        guess = best_next_guess(possible)


def best_next_guess(possible):
    """Pick the candidate (from all codes) minimizing the worst-case remaining possibilities."""
    possible_set = set(possible)
    best_candidate = None
    best_worst_case = None

    for candidate in ALL_CODES:
        bucket_sizes = Counter(score(candidate, secret_guess) for secret_guess in possible)
        worst_case = max(bucket_sizes.values())

        if best_worst_case is None or worst_case < best_worst_case or (
            worst_case == best_worst_case
            and best_candidate not in possible_set
            and candidate in possible_set
        ):
            best_worst_case = worst_case
            best_candidate = candidate

    return best_candidate

In [ ]:
# Demo: solve a single random secret, showing every guess.
secret = random_secret()
print(f"Secret (hidden from the solver): {format_code(secret)}\n")
guesses = solve(secret, verbose=True)
print(f"\nSolved in {len(guesses)} guesses.")

## Stats: solve a batch of random secrets

Knuth proved this strategy solves *every one* of the 1296 possible codes in at most 5 guesses, but brute-force testing all 1296 in pure Python is slow (each guess step re-evaluates every candidate code against every remaining possibility). This samples a batch of random secrets instead — enough to see the average and confirm the worst case never exceeds 5.

In [ ]:
sample_size = 60
sample = random.sample(ALL_CODES, sample_size)
results = [len(solve(secret)) for secret in sample]

print(f"Codes tested: {len(results)}")
print(f"Average guesses: {sum(results) / len(results):.3f}")
print(f"Worst case: {max(results)} guesses")
print(f"Distribution: {dict(sorted(Counter(results).items()))}")